# Norway Future Skills — Data Source Exploration

## Objective

This short notebook documents the **structure of the raw NAV job-posting data** before the cleaned dataset is analyzed.

Its purpose is deliberately narrow:

1. load the saved raw API response
2. inspect the top-level response structure
3. inspect the structure of an individual job posting
4. identify the nested fields that are useful for preprocessing
5. document the handoff from raw data to the cleaned analysis dataset

## What is intentionally not included here

Earlier versions of this notebook also contained:

- city-frequency analysis
- missing-location checks
- skill-frequency analysis
- manual AI/digital keyword searches
- creation of the exploded `job_skills.csv` table

Those tasks are now handled more carefully in:

- `01_data_quality_eda.ipynb`
- `02_skill_analysis.ipynb`
- `03_ai_future_skills.ipynb`

Keeping this notebook focused avoids duplicated analysis and gives the project a clearer workflow.

## Project data flow

```text
NAV Arbeidsplassen
        ↓
src/collect_nav_jobs.py
        ↓
data/raw/jobs_raw.json
        ↓
src/clean_jobs.py
        ↓
data/processed/jobs_clean.csv
        ↓
analysis notebooks
```


In [1]:
# Cell 02 — Import libraries and define the raw-data path

import json
from pathlib import Path

RAW_DATA_PATH = Path(
    "../data/raw/jobs_raw.json"
)

print(
    "Raw data file:",
    RAW_DATA_PATH
)

print(
    "File exists:",
    RAW_DATA_PATH.exists()
)


Raw data file: ..\data\raw\jobs_raw.json
File exists: True


## 1. Load the saved raw NAV response

The raw file contains the JSON response collected from NAV Arbeidsplassen.

We load the saved file rather than sending new API requests from the notebook. This keeps the analysis reproducible: rerunning the notebook inspects the same source data that was used in the project instead of whatever happens to be available from the API later.


In [2]:
# Cell 04 — Load the raw JSON file

with RAW_DATA_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    raw_data = json.load(file)

print(
    "Top-level keys:",
    list(raw_data.keys())
)


Top-level keys: ['took', 'timed_out', '_shards', 'hits', 'aggregations']


## 2. Inspect the API response structure

The saved response follows a search-result structure in which job postings are stored under:

```text
hits → hits
```

The surrounding response also contains search metadata such as the total number of matching records.

At this stage we are interested in understanding the structure, not performing labor-market analysis.


In [3]:
# Cell 06 — Inspect the search-result container

hits_container = raw_data.get(
    "hits",
    {}
)

raw_jobs = hits_container.get(
    "hits",
    []
)

print(
    "Keys inside 'hits':",
    list(hits_container.keys())
)

print(
    "Job records stored in this raw file:",
    len(raw_jobs)
)

print(
    "Reported total matches:",
    hits_container.get("total")
)


Keys inside 'hits': ['total', 'max_score', 'hits']
Job records stored in this raw file: 25
Reported total matches: {'value': 13914, 'relation': 'eq'}


### Important distinction

The number of records physically stored in this raw response is not necessarily the same as the total number of jobs available from the source.

The collection script is responsible for retrieving the larger project sample. The final cleaned dataset used in the analysis contains **10,000 job postings**.

This notebook therefore documents the raw response format; it is **not** the authoritative place for measuring dataset size or completeness.


## 3. Inspect one raw job posting

Each search hit contains metadata plus an `_source` object.

The `_source` object contains the fields that matter for preprocessing, such as:

- job identifier
- title
- employer/business name
- publication and expiration dates
- location information
- category and occupation information
- nested job properties


In [4]:
# Cell 09 — Inspect the first raw job record

if not raw_jobs:
    raise ValueError(
        "No job records were found in the raw JSON file."
    )

first_hit = raw_jobs[0]

print(
    "Search-hit keys:",
    list(first_hit.keys())
)

first_job = first_hit.get(
    "_source",
    {}
)

print(
    "\nFields inside '_source':"
)

print(
    list(first_job.keys())
)


Search-hit keys: ['_index', '_id', '_score', '_source', 'sort']

Fields inside '_source':
['expires', 'businessName', 'under18_facet', 'medium', 'source', 'published', 'title', 'uuid', 'generatedSearchMetadata', 'reference', 'locationList', 'categoryList', 'employer', 'occupationList', 'properties', 'status']


## 4. Inspect nested job properties

NAV stores several useful attributes inside the nested `properties` field.

The available keys can include information such as:

- work language
- generated search/skill tags
- education requirements
- application deadline
- job title
- employer
- remote-work information
- experience requirements
- driver's-license requirements

The exact properties available can vary by posting, so the cleaning process must handle missing nested values safely.


In [5]:
# Cell 11 — Inspect nested properties from the first posting

first_properties = (
    first_job.get(
        "properties",
        {}
    )
)

print(
    "Property keys:"
)

print(
    list(first_properties.keys())
)

print(
    "\nExample property values:"
)

for key, value in (
    first_properties.items()
):
    print(
        f"{key}: {value}"
    )


Property keys:
['workLanguage', 'searchtagsai', 'education', 'applicationdue', 'jobtitle', 'adtextFormat', 'employer', 'remote', 'experience', 'needDriversLicense', 'hasInterestform']

Example property values:
workLanguage: ['Norsk', 'Engelsk', 'Skandinavisk']
searchtagsai: ['Bygg', 'Byggtapetserer', 'Gulvbelegg', 'Gulvlegger', 'Håndverker', 'Malerarbeid', 'Renovering', 'Sparkling']
education: ['Ingen krav']
applicationdue: Snarest
jobtitle: Gulvlegger
adtextFormat: ikkeStrukturert
employer: Møller Håndverk As
remote: Hjemmekontor ikke mulig
experience: ['Noe']
needDriversLicense: ['true']
hasInterestform: true


## 5. Check the raw fields used by the project

The later analysis works with a simplified cleaned schema:

- `job_id`
- `title`
- `company`
- `published`
- `expires`
- `city`
- `occupation`
- `category`
- `skills`

The raw NAV response does not store all of these in a flat structure. Some values come from nested lists or properties.

The purpose of `src/clean_jobs.py` is to convert that source-specific JSON structure into a consistent tabular dataset while preserving explicit availability flags such as:

- `has_city`
- `has_occupation`
- `has_skills`

This separation is useful because **data extraction/cleaning** and **data analysis** are different stages of the workflow.


In [6]:
# Cell 13 — Confirm the main raw source fields are present

raw_field_check = {
    "job identifier (uuid)":
        "uuid" in first_job,
    "title":
        "title" in first_job,
    "business/employer information":
        (
            "businessName" in first_job
            or "employer" in first_job
        ),
    "published date":
        "published" in first_job,
    "expiration date":
        "expires" in first_job,
    "location list":
        "locationList" in first_job,
    "category list":
        "categoryList" in first_job,
    "occupation list":
        "occupationList" in first_job,
    "properties":
        "properties" in first_job
}

for field, available in (
    raw_field_check.items()
):
    print(
        f"{field}: {available}"
    )


job identifier (uuid): True
title: True
business/employer information: True
published date: True
expiration date: True
location list: True
category list: True
occupation list: True
properties: True


## 6. Preprocessing handoff

This notebook stops at **source understanding**.

The reproducible processing pipeline is handled by project scripts rather than exploratory notebook code:

### Collection
`src/collect_nav_jobs.py`

Retrieves job-posting records from the source and saves the raw response.

### Cleaning
`src/clean_jobs.py`

Transforms the nested source structure into:

`data/processed/jobs_clean.csv`

### Analysis
The cleaned file is then used by the analytical notebooks.

This design avoids mixing API experiments, preprocessing, data-quality checks, and final analysis in one notebook.


In [7]:
# Cell 15 — Verify the cleaned analysis dataset is available

CLEAN_DATA_PATH = Path(
    "../data/processed/jobs_clean.csv"
)

print(
    "Cleaned dataset:",
    CLEAN_DATA_PATH
)

print(
    "Cleaned dataset exists:",
    CLEAN_DATA_PATH.exists()
)

if CLEAN_DATA_PATH.exists():
    print(
        "\nThe preprocessing stage is complete. "
        "Continue with 01_data_quality_eda.ipynb "
        "for dataset validation and EDA."
    )


Cleaned dataset: ..\data\processed\jobs_clean.csv
Cleaned dataset exists: True

The preprocessing stage is complete. Continue with 01_data_quality_eda.ipynb for dataset validation and EDA.


# Summary

This notebook establishes where the project data comes from and how the raw source is structured.

## Key points

- NAV job postings are collected as nested JSON search results.
- Individual postings are stored inside the `_source` object.
- Important analytical fields originate from several different raw structures, including nested lists and `properties`.
- The notebook reads the **saved raw response** rather than making live API requests, improving reproducibility.
- Data collection and cleaning are delegated to dedicated scripts.
- Dataset quality, geographic analysis, skill extraction, and AI/digital classification are intentionally left to the later notebooks.

The next notebook, `01_data_quality_eda.ipynb`, begins with the cleaned 10,000-job dataset and evaluates whether its fields are suitable for analysis.
